In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("housing.csv")
df['income_cat'] = pd.cut(df["median_income"], bins = [0, 1.5, 3, 4.5, 6, np.inf], labels= [1,2,3,4,5])
df

In [ ]:
import matplotlib.pyplot as plt
df["income_cat"].value_counts().sort_index().plot.bar(rot=0, grid=True)
plt.title("Income Categories Distribution")
plt.xlabel("Income Category")

In [ ]:
df.describe()

In [ ]:
import matplotlib as plt
df.hist(bins=50, figsize=(12,8))

In [ ]:
import numpy as np

In [ ]:
# def shuffle_and_split(data, test_ratio):
#     np.random.seed(42)
#     shuffle_indices = np.random.permutation(len(data))
#     # print(shuffle_indices)
#     test_set_size = int(len(data) * test_ratio)
#     test_indices = shuffle_indices[: test_set_size]
#     train_indices = shuffle_indices[test_set_size: ]
#     return data.iloc[train_indices], data.iloc[test_indices]


In [ ]:
# train, test = shuffle_and_split(df,0.4)

In [ ]:
# train

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(df, df['income_cat']):
    df.loc[test_index].drop('income_cat', axis=1).to_csv("input_1.csv", index=False)
    strait_train_set = df.loc[train_index]
    strait_test_set = df.loc[test_index]


In [ ]:
import matplotlib.pyplot as plt
strait_train_set["income_cat"].value_counts().sort_index().plot.bar(rot=0, grid=True)
plt.title("Income Categories Distribution")
plt.xlabel("Income Category")

In [ ]:
import matplotlib.pyplot as plt
strait_test_set["income_cat"].value_counts().sort_index().plot.bar(rot=0, grid=True)
plt.title("Income Categories Distribution")
plt.xlabel("Income Category")

In [ ]:
df = strait_train_set.copy()
df

In [ ]:
housing = strait_train_set.drop("median_house_value", axis=1).copy()
housing_lables = strait_train_set["median_house_value"].copy()


In [ ]:
housing_lables

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')

In [ ]:
housing_num = housing.select_dtypes(include=[np.number])
X = imputer.fit_transform(housing_num)

In [ ]:
X

In [ ]:
housing

In [ ]:
housing = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index)
housing

In [ ]:
housing["ocean_proximity"] = df["ocean_proximity"]
housing

In [ ]:
housing = housing[['ocean_proximity']]

In [ ]:
set(housing['ocean_proximity'])


In [ ]:
# from sklearn.preprocessing import OrdinalEncoder
# ordinal_encoder = OrdinalEncoder()
# housing_cat = ordinal_encoder.fit_transform(housing)

In [ ]:
# housing_cat

In [ ]:
# housing_cat = pd.DataFrame(housing_cat, columns=housing.columns, index=housing.index)

In [ ]:
# housing_cat
#ordinal encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder = OneHotEncoder()
housing_cat = one_hot_encoder.fit_transform(housing)

In [ ]:
# housing

In [ ]:
housing_cat

In [ ]:
housing_cat.toarray()


In [ ]:
housing_cat = pd.DataFrame(housing_cat.toarray(), columns=['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'], index=housing.index)

In [ ]:
housing_cat

In [ ]:
df

In [ ]:
df

In [ ]:
Income_cat = df["income_cat"].copy()


In [ ]:
df = df.drop(["ocean_proximity", "income_cat", "median_house_value"], axis=1, errors="ignore")


In [ ]:
df

In [ ]:
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()
df_scaled = scalar.fit_transform(df)


In [ ]:
df_scaled

In [ ]:
df_scaled = pd.DataFrame(df_scaled, columns=df.columns, index=df.index)

In [ ]:
df_scaled

In [ ]:
df = pd.concat([df_scaled, housing_cat], axis=1)
df = pd.concat([df, Income_cat], axis=1)


In [ ]:
df

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

model = RandomForestRegressor(random_state=42)
model.fit(df, housing_lables)


In [ ]:
input_data = pd.read_csv("input_1.csv")
input_df = input_data.copy()
input_df["income_cat"] = pd.cut(input_df["median_income"], bins=[0, 1.5, 3, 4.5, 6, np.inf], labels=[1,2,3,4,5])


In [ ]:
input_num = input_df.drop(["ocean_proximity", "income_cat", "median_house_value"], axis=1, errors="ignore")
input_num_imputed = imputer.transform(input_num)
input_num_df = pd.DataFrame(input_num_imputed, columns=input_num.columns, index=input_df.index)


In [ ]:
input_num

In [ ]:
input_num_df

In [ ]:
input_cat_encoded = one_hot_encoder.transform(input_df[["ocean_proximity"]])
input_cat_df = pd.DataFrame(input_cat_encoded.toarray(), columns=["<1H OCEAN", "INLAND", "ISLAND", "NEAR BAY", "NEAR OCEAN"], index=input_df.index)


In [ ]:
input_scaled = scalar.transform(input_num_df)
scaled_df = pd.DataFrame(input_scaled, columns=input_num_df.columns, index=input_df.index)

In [ ]:
X_input = pd.concat([scaled_df, input_cat_df, input_df["income_cat"]], axis=1)
X_input = X_input[model.feature_names_in_]


In [ ]:
X_input


In [ ]:
predictions = model.predict(X_input)


In [ ]:
input_data["median_house_value"] = predictions
input_data.to_csv("output_1.csv", index=False)
print("Inference is Done! Results saved to output_1.csv")
